# 2.4 Training an ExtraTrees Model for Reservoir Storage Forecasting

**STARS4Water – Training Video 2.4**
*UK Centre for Ecology and Hydrology*

This notebook accompanies the training video *Machine Learning model for reservoir storage forecasting*, available at the [STARS4Water Academy](https://stars4water.eu/the-stars4water-academy/)

It demonstrates how to **train** an ExtraTrees machine learning model for 1-month ahead reservoir storage forecasting using data from 35 reservoirs across the UK, Spain and France.

**What this notebook covers:**
1. Required packages
2. Load and explore the data
3. Train an ExtraTrees model
4. Evaluate the model
5. Hyperparameter tuning, feature selection and overfitting

---
### Running this notebook

**Option 1 – Binder (no installation required)**  
Click the Binder badge in the repository to launch this notebook directly in your browser. All dependencies are installed automatically.

**Option 2 – Local installation**  
Create a virtual environment and install dependencies:

```bash
python3 -m venv venv
source venv/bin/activate          # Windows: venv\Scripts\activate
pip install --upgrade pip
pip install -r requirements.txt
```

Register the kernel and launch Jupyter:

```bash
python -m ipykernel install --user --name=venv --display-name "Python (venv)"
jupyter notebook
```

> **Python version note:** `scikit-learn ≥ 1.7` requires **Python 3.10 or later**.

## 1. Import packages

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_absolute_error

print("Packages loaded successfully.")

## 2. Load and explore the data

`data.sav` is a pickled pandas DataFrame containing monthly records for **35 reservoirs** across the UK, Spain and France. All storage values are expressed as a **percentage of capacity**.

| Feature group | Features | Description |
|---|---|---|
| **Target** | `storage` | Reservoir storage at time *T* (%) |
| **Lagged storage** | `storage1`, `storage2` | Storage 1 and 2 months ago (%) |
| **Rainfall** | `rainfall`, `rainfall1` | Monthly rainfall (mm): current, 1 month ago |
| **Cumulative rainfall** | `rfquart`, `rfhalf` | Cumulative rainfall over last 3 and 6 months (mm) |
| **Seasonality** | `datesin`, `datecos` | Cyclical month encoding: sin and cos of (2π × month / 12) |
| **Climate** | `swc_pc_s`, `high_prec_freq`, `low_prec_dur` | Soil water content, high precipitation frequency, low precipitation duration |
| **Catchment** | `run_mm_syr`, `ele_mt_smx` | Long-term runoff, elevation |
| **Reservoir** | `Inception_use_WR/P/Irr/FC`, `Catchment_area`, `Capacity` | Purpose flags, catchment area, capacity |

Catchment and climate features from the [Caravan](https://www.nature.com/articles/s41597-023-01975-w) and [CAMELS](https://essd.copernicus.org/articles/17/1461/2025/) datasets.

In [ ]:
DATA_PATH = 'data.sav'
with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f)

n_res   = data['RESERVOIR_ID'].nunique()
n_rec   = len(data)
date_lo = data['date'].min().strftime('%b %Y')
date_hi = data['date'].max().strftime('%b %Y')

print(f"Reservoirs : {n_res}")
print(f"Records    : {n_rec}")
print(f"Period     : {date_lo} – {date_hi}")

In [ ]:
data

### Feature engineering

The features in `data.sav` have been pre-engineered from monthly reservoir and catchment records. If you are working with your own raw data, the key transformations are shown below.

**Lagged storage** captures how current storage carries over month to month — the most important predictor in the model.

**Lagged and cumulative rainfall** looks at current and antecedant rainfall, including rolling means.

**Cyclical month encoding** uses sine and cosine rather than a raw month number, so the model sees December and January as adjacent (no discontinuity at the year boundary).

**one-hot-encoding** converts categorical variables to numeric forms that the model can use.

In [ ]:
# Feature engineering — already applied in data.sav.
# Replicate these steps when preparing your own dataset.

# df = df.sort_values(['RESERVOIR_ID', 'date'])

# Lagged storage
# df['storage1'] = df.groupby('RESERVOIR_ID')['storage'].shift(1)
# df['storage2'] = df.groupby('RESERVOIR_ID')['storage'].shift(2)

# Lagged and cumulative rainfall
# df['rainfall1'] = df.groupby('RESERVOIR_ID')['rainfall'].shift(1)
# df['rfquart']   = df.groupby('RESERVOIR_ID')['rainfall'].transform(lambda x: x.rolling(3).sum())
# df['rfhalf']    = df.groupby('RESERVOIR_ID')['rainfall'].transform(lambda x: x.rolling(6).sum())

# Cyclical month encoding
# df['datesin'] = np.sin(2 * np.pi * df['date'].dt.month / 12)
# df['datecos'] = np.cos(2 * np.pi * df['date'].dt.month / 12)

## 3. Train the model

We use a **temporal train/test split**: data up to the end of 2014 are used for training; 2015 onwards are held back for evaluation. This mirrors a realistic operational setting where the model is trained on historical observations and applied to future periods.

**Note**: due to the difference in timeseries length for each reservoir, this means that some reservoirs contribute more to training - ideally the input would be balanced.

In [ ]:
FEATURE_NAMES = [
    'storage1', 'storage2',
    'rainfall', 'rainfall1', 'rfquart', 'rfhalf',
    'datesin', 'datecos',
    'swc_pc_s', 'high_prec_freq', 'low_prec_dur',
    'run_mm_syr', 'ele_mt_smx',
    'Inception_use_WR', 'Inception_use_P', 'Inception_use_Irr', 'Inception_use_FC',
    'Catchment_area', 'Capacity',
]
TARGET     = 'storage'
SPLIT_YEAR = 2015

data_clean = data.dropna(subset=FEATURE_NAMES + [TARGET])
train = data_clean[data_clean['date'].dt.year <  SPLIT_YEAR]
test  = data_clean[data_clean['date'].dt.year >= SPLIT_YEAR]

X_train, y_train = train[FEATURE_NAMES].values, train[TARGET].values
X_test,  y_test  = test[FEATURE_NAMES].values,  test[TARGET].values

print(f"Training records : {len(X_train):,}  (up to Dec {SPLIT_YEAR - 1})")
print(f"Test records     : {len(X_test):,}   ({SPLIT_YEAR} onwards)")

##### The ExtraTrees hyperparameters used here match the pretrained model from Video 2.3. Section 5 discusses how to tune them for your own dataset.

In [ ]:
model = ExtraTreesRegressor(
    n_estimators=200,
    max_depth=15,
    max_features=20,
    min_samples_leaf=6,
    bootstrap=True,
    random_state=345,
    n_jobs=-1,
)

model.fit(X_train, y_train)

## 4. Evaluate the model

We evaluate the trained model on the held-out test set (2015–2022), both overall and per reservoir.

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

# Per-reservoir R² on test set
results = []
for res_id, grp in test.groupby('RESERVOIR_ID'):
    y_o = grp[TARGET].values
    y_p = model.predict(grp[FEATURE_NAMES].values)
    results.append({'RESERVOIR_ID': res_id, 'R2': r2_score(y_o, y_p), 'MAE': mean_absolute_error(y_o, y_p)})

results_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: observed vs predicted
axes[0].scatter(y_test, y_test_pred, alpha=0.3, s=8, color='steelblue', edgecolors='none')
axes[0].plot([0, 105], [0, 105], color='firebrick', linewidth=1.5, linestyle='--', label='1:1 line')
axes[0].set_xlabel('Observed Storage (%)', fontsize=11)
axes[0].set_ylabel('Predicted Storage (%)', fontsize=11)
r2_all  = r2_score(y_test, y_test_pred)
mae_all = mean_absolute_error(y_test, y_test_pred)
axes[0].set_title(f'Test set: all reservoirs\nR² = {r2_all:.3f},  MAE = {mae_all:.1f} pp', fontsize=12)
axes[0].set_xlim(0, 105)
axes[0].set_ylim(0, 105)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.25)

# Box plot of per-reservoir R² distribution
axes[1].boxplot(
    results_df['R2'],
    vert=True,
    patch_artist=True,
    boxprops=dict(facecolor='steelblue', alpha=0.6),
    medianprops=dict(color='firebrick', linewidth=2),
    flierprops=dict(marker='o', markerfacecolor='steelblue', markersize=5, alpha=0.6),
)
axes[1].set_ylabel('R²', fontsize=11)
axes[1].set_xticks([1])
axes[1].set_xticklabels([f'n = {len(results_df)} reservoirs'], fontsize=10)
axes[1].set_title(
    f'Per-reservoir R² distribution (test set)\nMedian = {results_df["R2"].median():.3f}',
    fontsize=12,
)
axes[1].axhline(0.85, color='grey', linestyle=':', linewidth=1, label='R² = 0.85')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.25, axis='y')

plt.tight_layout()
plt.show()

print(f"Median R²: {results_df['R2'].median():.2f},  MAE: {results_df['MAE'].median():.1f} pp")
print(f"Min R²   : {results_df['R2'].min():.2f}")
print(f"Max R²   : {results_df['R2'].max():.2f}")


### Save the trained model

The trained model is saved as a pickle file and can be loaded directly in the application notebook (Video 2.3).

In [ ]:
MODEL_OUT = 'model_trained.sav'
with open(MODEL_OUT, 'wb') as f:
    pickle.dump(model, f)
print(f"Model saved to {MODEL_OUT}")

# To reload:
# with open(MODEL_OUT, 'rb') as f:
#     model = pickle.load(f)

## 5. Hyperparameter tuning, feature selection and overfitting

### Overfitting

Tree-based models can memorise training data, producing near-perfect training scores but poor generalisation to new data. Comparing training and test performance reveals the degree of overfitting — a large gap between the two signals that the model is too complex for the available data.

In [ ]:
def per_reservoir_metrics(df, split_label):
    rows = []
    for res_id, grp in df.groupby('RESERVOIR_ID'):
        y_o = grp[TARGET].values
        y_p = model.predict(grp[FEATURE_NAMES].values)
        rows.append({'R2': r2_score(y_o, y_p), 'MAE': mean_absolute_error(y_o, y_p)})
    tmp = pd.DataFrame(rows)
    return {'Set': split_label, 'Median R²': tmp['R2'].median(), 'Median MAE': tmp['MAE'].median()}

metrics = pd.DataFrame([
    per_reservoir_metrics(train, 'Train'),
    per_reservoir_metrics(test,  'Test'),
]).round(2)
print(metrics.to_string(index=False))


### Feature selection

Feature importances show how much each variable contributes to the model's predictions. Low-importance features can be removed without significantly reducing performance — simplifying the model and reducing the data requirements for users applying it to new regions.

More information on feature selection can be found here: https://scikit-learn.org/stable/modules/feature_selection.html

### Hyperparameter tuning

The key hyperparameters for ExtraTrees are:

| Hyperparameter | Description | Typical range |
|---|---|---|
| `n_estimators` | Number of trees | 100–500 |
| `max_depth` | Maximum tree depth — limits overfitting | 5–30, or `None` |
| `max_features` | Features considered at each split — controls tree diversity | 5–`n_features` |
| `min_samples_leaf` | Minimum samples per leaf — limits overfitting | 1–10 |
| `bootstrap` | Resample training data for each tree | `True` / `False` |

`RandomizedSearchCV` efficiently explores this space by sampling a fixed number of parameter combinations, rather than exhaustively testing all of them. Uncomment the cell below to run a search.

In [ ]:
# Hyperparameter search — uncomment to run (may take several minutes)

# from sklearn.model_selection import RandomizedSearchCV

# param_distributions = {
#     'n_estimators'    : [100, 200, 300],
#     'max_depth'       : [10, 15, 20, None],
#     'max_features'    : [10, 15, 20],
#     'min_samples_leaf': [3, 6, 9],
# }

# search = RandomizedSearchCV(
#     ExtraTreesRegressor(bootstrap=True, random_state=345, n_jobs=-1),
#     param_distributions,
#     n_iter=20,
#     scoring='r2',
#     cv=3,
#     random_state=345,
#     verbose=1,
# )
# search.fit(X_train, y_train)
# print("Best parameters:", search.best_params_)
# print(f"Best CV R²      : {search.best_score_:.3f}")


> **Further reading:** For full technical details on the model and training methodology, see the **STARS4Water Deliverable D3.4 report** available on the [STARS4Water project website](https://stars4water.eu/).